# Exit intention: old MD set vs. Kimi-relabeled MD set

Compares the exit-intention distribution for two groups:

- **Old MD** — all posts BAT originally flagged `MD == YES` (`md_only.csv`, 3,394 rows)
- **New MD** — the subset of those Kimi re-confirmed as `MD == YES` (`md_only_relabeled.csv`, `MD_llm` column)

Also breaks out the **flipped** posts (BAT said MD=YES, Kimi said MD=NO) as their own group, since
that's the more diagnostic comparison for whether disagreement cases look systematically different
on exit intent.

Needs `exit_annotated_pass1_merged.csv` for the exit-intention labels.

## Config + load

In [13]:
import pandas as pd
from scipy.stats import chi2_contingency

MD_ONLY_CSV = "md_only.csv"
MD_RELABELED_CSV = "md_only_relabeled.csv"
EXIT_CSV = "/Users/nadia/Desktop/redditRun_june/EXIT_slurm/exit_annotated_pass1_merged.csv"

def find_col(df, candidates, label, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not find a {label} column. Columns present: {list(df.columns)}")
    return None

old_md = pd.read_csv(MD_ONLY_CSV)
new_md = pd.read_csv(MD_RELABELED_CSV)
exit_df = pd.read_csv(EXIT_CSV)

print(f"old_md (BAT MD==YES):        {len(old_md):,} rows")
print(f"new_md (relabeled, has MD_llm): {len(new_md):,} rows")
print(f"exit_df:                     {len(exit_df):,} rows")
print("\nexit_df columns:", list(exit_df.columns))

old_md (BAT MD==YES):        3,394 rows
new_md (relabeled, has MD_llm): 3,394 rows
exit_df:                     144,652 rows

exit_df columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'exit_intention_class', 'exit_confidence', 'exit_evidence_span', 'exit_reasoning', 'exit_level', 'exit_reason_primary', 'exit_reason_secondary']


## Detect id + exit-label columns

Check the printed column name for the exit label below — if the auto-detect list doesn't match,
add the real column name to the candidates list and re-run.

In [14]:
id_col = find_col(old_md, ["id", "post_id"], "id")

EXIT_LABEL_CANDIDATES = ["exit_intention_class"]
exit_id_col = find_col(exit_df, ["id", "post_id"], "id")
exit_col = find_col(exit_df, EXIT_LABEL_CANDIDATES, "exit label")

print(f"id column (BAT files): '{id_col}'")
print(f"id column (exit file): '{exit_id_col}'")
print(f"exit label column: '{exit_col}'")
print("\nRaw exit label value counts:")
print(exit_df[exit_col].value_counts(dropna=False).to_string())

id column (BAT files): 'post_id'
id column (exit file): 'post_id'
exit label column: 'exit_intention_class'

Raw exit label value counts:
exit_intention_class
no_exit               137779
exit_contemplating      5354
exit_explicit           1519


## Build the three groups and merge in exit labels

- `old_md` → all originally-flagged MD posts
- `new_md_confirmed` → BAT MD=YES **and** Kimi MD=YES
- `new_md_flipped` → BAT MD=YES **but** Kimi MD=NO

In [15]:
for df in (old_md, new_md):
    df[id_col] = df[id_col].astype(str)
exit_df[exit_id_col] = exit_df[exit_id_col].astype(str)

# normalize MD_llm to YES/NO uppercase for a clean split
md_llm_norm = new_md["MD_llm"].astype(str).str.strip().str.upper()
new_md_confirmed = new_md[md_llm_norm == "YES"].copy()
new_md_flipped = new_md[md_llm_norm == "NO"].copy()

print(f"new_md_confirmed (BAT YES, Kimi YES): {len(new_md_confirmed):,}")
print(f"new_md_flipped   (BAT YES, Kimi NO):  {len(new_md_flipped):,}")
unresolved = len(new_md) - len(new_md_confirmed) - len(new_md_flipped)
if unresolved:
    print(f"NOTE: {unresolved} rows had neither YES nor NO in MD_llm (parse errors) — check separately.")

exit_lookup = exit_df[[exit_id_col, exit_col]].rename(
    columns={exit_id_col: id_col, exit_col: "exit_label"}
)

def attach_exit(df, name):
    merged = df.merge(exit_lookup, on=id_col, how="left")
    missing = merged["exit_label"].isna().sum()
    print(f"{name}: {len(merged):,} rows, {missing:,} missing an exit label after merge ({missing/len(merged):.1%})")
    return merged

old_md_x = attach_exit(old_md, "old_md")
new_md_confirmed_x = attach_exit(new_md_confirmed, "new_md_confirmed")
new_md_flipped_x = attach_exit(new_md_flipped, "new_md_flipped")

new_md_confirmed (BAT YES, Kimi YES): 2,399
new_md_flipped   (BAT YES, Kimi NO):  995
old_md: 3,394 rows, 0 missing an exit label after merge (0.0%)
new_md_confirmed: 2,399 rows, 0 missing an exit label after merge (0.0%)
new_md_flipped: 995 rows, 0 missing an exit label after merge (0.0%)


## Exit intention distribution by group

In [16]:
def dist_table(df, name):
    counts = df["exit_label"].value_counts(dropna=False)
    pct = df["exit_label"].value_counts(normalize=True, dropna=False) * 100
    out = pd.DataFrame({"n": counts, "pct": pct.round(1)})
    out.columns = pd.MultiIndex.from_product([[name], out.columns])
    return out

comparison = pd.concat(
    [
        dist_table(old_md_x, "old_md (all BAT MD=YES)"),
        dist_table(new_md_confirmed_x, "new_md_confirmed (Kimi relabeled)"),
        dist_table(new_md_flipped_x, "new_md_flipped (relabeled disagrees)"),
    ],
    axis=1,
).fillna(0)

comparison

old_md (all BAT MD=YES)        \
                                         n   pct   
exit_label                                         
no_exit                               1690  49.8   
exit_contemplating                    1307  38.5   
exit_explicit                          397  11.7   

                   new_md_confirmed (Kimi relabeled)        \
                                                   n   pct   
exit_label                                                   
no_exit                                         1140  47.5   
exit_contemplating                               956  39.8   
exit_explicit                                    303  12.6   

                   new_md_flipped (relabeled disagrees)        
                                                      n   pct  
exit_label                                                     
no_exit                                             550  55.3  
exit_contemplating                                  351  35.3  
exit_explicit                                        94   9.4

## Does it change meaningfully? — chi-square + Cramer's V

Two tests:
1. **old_md vs new_md_confirmed** — does restricting to Kimi-confirmed posts shift the exit-intent mix?
2. **new_md_confirmed vs new_md_flipped** — the more diagnostic one: do the posts Kimi *disagreed* on
   look different on exit intent than the ones it confirmed? (Cohen 1988 benchmarks: V=0.10 small,
   0.30 medium, 0.50 large — already your convention elsewhere in the paper.)

In [17]:

    
def compare_two(df_a, name_a, df_b, name_b):
    a = df_a["exit_label"].dropna().reset_index(drop=True)
    b = df_b["exit_label"].dropna().reset_index(drop=True)
    labels = pd.concat([a, b], ignore_index=True)
    groups = pd.Series([name_a] * len(a) + [name_b] * len(b), name="group")
    table = pd.crosstab(labels, groups)
    chi2, p, dof, _ = chi2_contingency(table)
    n = table.values.sum()
    r, k = table.shape
    cramers_v = (chi2 / (n * (min(r, k) - 1))) ** 0.5
    print(f"{name_a} (n={len(a)}) vs {name_b} (n={len(b)})")
    print(f"  chi2={chi2:.3f}, dof={dof}, p={p:.4f}, Cramer's V={cramers_v:.3f}")
    print(table)
    print()
    return chi2, p, cramers_v

print("=== Test 1: old_md vs new_md_confirmed ===")
compare_two(old_md_x, "old_md", new_md_confirmed_x, "new_md_confirmed")

print("=== Test 2: new_md_confirmed vs new_md_flipped ===")
compare_two(new_md_confirmed_x, "new_md_confirmed", new_md_flipped_x, "new_md_flipped")
    
    
    

=== Test 1: old_md vs new_md_confirmed ===
old_md (n=3394) vs new_md_confirmed (n=2399)
  chi2=3.147, dof=2, p=0.2073, Cramer's V=0.023
group               new_md_confirmed  old_md
exit_label                                  
exit_contemplating               956    1307
exit_explicit                    303     397
no_exit                         1140    1690

=== Test 2: new_md_confirmed vs new_md_flipped ===
new_md_confirmed (n=2399) vs new_md_flipped (n=995)
  chi2=18.410, dof=2, p=0.0001, Cramer's V=0.074
group               new_md_confirmed  new_md_flipped
exit_label                                          
exit_contemplating               956             351
exit_explicit                    303              94
no_exit                         1140             550



(18.40978100424143, 0.00010054647474108222, 0.07364926078044917)

## Exit-explicit / exit-contemplating rate (collapsed to any-exit vs no-exit)

If the exit label has more than the three named categories, this collapses to a binary
"any exit intent" vs "no exit intent" view, which is usually the more paper-ready number.
Adjust the `no_exit_values` list below to match your file's actual category names.

In [9]:
no_exit_values = {"no_exit", "no exit", "none", "no"}  # adjust to match your actual label strings

def any_exit_rate(df, name):
    labels = df["exit_label"].dropna().astype(str).str.strip().str.lower()
    any_exit = (~labels.isin(no_exit_values)).mean()
    print(f"{name}: {any_exit:.1%} have any exit intent (n={len(labels)})")
    return any_exit

any_exit_rate(old_md_x, "old_md")
any_exit_rate(new_md_confirmed_x, "new_md_confirmed")
any_exit_rate(new_md_flipped_x, "new_md_flipped")

old_md: 50.2% have any exit intent (n=3394)
new_md_confirmed: 52.5% have any exit intent (n=2395)
new_md_flipped: 44.7% have any exit intent (n=991)


0.44702320887991925

['d1ss5w', 'brcwqn', 'dmlbn5', 'dgie7z', 'dd8nvi', 'cicjdq', 't8n9y4', 'w9gy3k']


NameError: name 'run_batch' is not defined